# Notebook 1: Setting up Setting A and Setting B

**Goal of this notebook:** decide, once and for all, *which image belongs to which role* (train, validation,
calibration, test, unknown) and save that decision in one table. Every later notebook will read this table
instead of re-splitting the data, so all methods are compared on exactly the same images.

**No model is trained here.** We only organise data. We go one small step at a time, and after each step
we print something so you can check it with your own eyes.

### The two settings in one minute
| | Setting A | Setting B |
|---|---|---|
| Field (PlantDoc) labels available? | none | a small labelled sample (~10%) |
| Used to calibrate confidence | PlantVillage validation images | that small PlantDoc sample |
| Field test set | all PlantDoc images of known classes | the remaining ~90% |

*Calibration* here means tuning how confident the model is (we do this later with temperature scaling).
Think of the calibration set as images we use to adjust the model's confidence, which must never be the
same images we use to judge it.

### Plan for this notebook
1. Settings and imports
2. Look at the folders (check before trusting)
3. Match class names between the two datasets
4. Split PlantVillage into train / validation / test
5. Separate PlantDoc into known and unknown classes
6. Build Setting A
7. Build Setting B
8. Safety checks
9. Save the table

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

In [2]:
# ---- Edit these -------------------------------------------------------------
PV_ROOT = Path(r"F:\mphil\pv")   # folder holding the PlantVillage images
PD_ROOT = Path(r"F:\mphil\pdoc")       # folder holding the PlantDoc leaf crops
OUT_DIR = Path("outputs")             # where we save results

# How many folders deep is the class folder?
#   0 -> ROOT/<class>/image.jpg                (PlantVillage is usually like this)
#   1 -> ROOT/train/<class>/image.jpg          (PlantDoc is usually like this: train/ and test/ first)
# Step 2 prints a hint if it thinks you picked the wrong level.
PV_CLASS_LEVEL = 0
PD_CLASS_LEVEL = 1

SEEDS = list(range(42, 52))   # Setting B is repeated 10 times (seeds 42 to 51), like the reference study
CAL_FRAC = 0.10               # about 10% of PlantDoc known images become the calibration set
PV_SEED = 42                  # random seed for the PlantVillage split
# -----------------------------------------------------------------------------

OUT_DIR.mkdir(exist_ok=True)

## Step 2: Look at the folders

Before splitting anything, we list every image in both datasets and count them per class folder.
A *random seed* (like 42) just makes "random" choices repeatable, so you and I get the same split.

In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


def list_images(root, class_level):
    """Return a table with one row per image: its path (relative to root) and its class folder name."""
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
            relative = p.relative_to(root)
            rows.append({"path": relative.as_posix(),
                         "class_folder": relative.parts[class_level],
                         # the folder(s) above the class folder, e.g. "train" or "test" ("" if none)
                         "original_subset": "/".join(relative.parts[:class_level])})
    return pd.DataFrame(rows)


pv_all = list_images(PV_ROOT, PV_CLASS_LEVEL)
pd_all = list_images(PD_ROOT, PD_CLASS_LEVEL)

print(f"PlantVillage: {len(pv_all)} images in {pv_all['class_folder'].nunique()} class folders")
print(f"PlantDoc:     {len(pd_all)} images in {pd_all['class_folder'].nunique()} class folders")

PlantVillage: 54305 images in 38 class folders
PlantDoc:     2484 images in 28 class folders


In [4]:
# Images per class folder, side by side
print("PlantVillage folders:")
print(pv_all["class_folder"].value_counts().sort_index().to_string())
print()
print("PlantDoc folders:")
print(pd_all["class_folder"].value_counts().sort_index().to_string())

PlantVillage folders:
class_folder
Apple___Apple_scab                                     630
Apple___Black_rot                                      621
Apple___Cedar_apple_rust                               275
Apple___healthy                                       1645
Blueberry___healthy                                   1502
Cherry_(including_sour)___Powdery_mildew              1052
Cherry_(including_sour)___healthy                      854
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     513
Corn_(maize)___Common_rust_                           1192
Corn_(maize)___Northern_Leaf_Blight                    985
Corn_(maize)___healthy                                1162
Grape___Black_rot                                     1180
Grape___Esca_(Black_Measles)                          1383
Grape___Leaf_blight_(Isariopsis_Leaf_Spot)            1076
Grape___healthy                                        423
Orange___Haunglongbing_(Citrus_greening)              5507
Peach___Bacterial_spo

### Check the folder layout

The two datasets are organised differently, and our code must read both correctly:

* **PlantDoc** is usually split into `train/` and `test/`, each containing one folder per class.
  The class folder is then **one level down** (`PD_CLASS_LEVEL = 1`).
* **PlantVillage** usually has just one folder per class and no train/test split
  (`PV_CLASS_LEVEL = 0`). Some downloads contain several versions (`color`, `grayscale`, `segmented`).
  If yours does, point `PV_ROOT` at the **color** folder only, so the versions are not mixed.

The cell below prints the top-level folders of each dataset and warns if the setting looks wrong.

In [5]:
for name, table, root, level in [("PlantVillage", pv_all, PV_ROOT, PV_CLASS_LEVEL),
                                 ("PlantDoc", pd_all, PD_ROOT, PD_CLASS_LEVEL)]:
    top = sorted(table["path"].str.split("/").str[0].unique())
    print(f"{name}: top-level folders = {top[:8]}{' ...' if len(top) > 8 else ''}")
    if level == 0 and set(top) <= {"train", "test", "val"}:
        print("   -> These look like train/test folders. Set the CLASS_LEVEL to 1 in Step 1 and re-run.")
    if level == 0 and {"color", "grayscale", "segmented"} & set(top):
        print("   -> Several image versions found. Point the ROOT at the color folder only.")
    if level == 1 and set(top) - {"train", "test", "val"}:
        print("   -> These do not look like train/test folders. Is the CLASS_LEVEL = 1 setting right?")

print()
print("PlantDoc images per original subset:")
print(pd_all["original_subset"].value_counts().to_string())

PlantVillage: top-level folders = ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot'] ...
PlantDoc: top-level folders = ['test', 'train']

PlantDoc images per original subset:
original_subset
train    2253
test      231


**Checkpoint.** Do the numbers look reasonable? (The reference study's PlantVillage has over 54,000 images;
its PlantDoc crops total 2,571.) If a count is zero or very different, fix the path or `*_CLASS_LEVEL`
before continuing.

## Step 3: Match class names between the datasets

PlantVillage and PlantDoc name the same disease differently, and only some diseases exist in both.
We need a small table that says "this PlantVillage folder and this PlantDoc folder are the same class".

Create a file called `class_alignment.csv` next to this notebook with three columns:

```
canonical,pv_folder,pd_folder
Tomato_late_blight,Tomato___Late_blight,Tomato leaf late blight
...
```

* `canonical` is a name you choose for the class (used everywhere later).
* `pv_folder` and `pd_folder` must **exactly** match the folder names printed in Step 2.
* The reference study found **21** shared classes. Use its Supplementary Table 1 as your guide.

The line above is only an illustration of the format. Do not copy it unless it matches your folders.

In [7]:
alignment = pd.read_csv("class_alignment_template.csv")
alignment

,canonical,pv_folder,pd_folder
0,Apple|Healthy,Apple___healthy,Apple leaf
1,Grape|Healthy,Grape___healthy,grape leaf
2,Peach|Healthy,Peach___healthy,Peach leaf
3,Strawberry|Healthy,Strawberry___healthy,Strawberry leaf
4,Tomato|Septoria Leaf Spot,Tomato___Septoria_leaf_spot,Tomato Septoria leaf spot
5,Tomato|Healthy,Tomato___healthy,Tomato leaf
6,Apple|Apple Scab,Apple___Apple_scab,Apple Scab Leaf
7,Apple|Cedar Apple Rust,Apple___Cedar_apple_rust,Apple rust leaf
8,Corn|Cercospora Leaf Spot Gray Leaf Spot,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,Corn Gray leaf spot
9,Corn|Common Rust,Corn_(maize)___Common_rust_,Corn rust leaf


In [8]:
# Make sure every folder named in the table really exists on disk
missing_pv = set(alignment["pv_folder"]) - set(pv_all["class_folder"])
missing_pd = set(alignment["pd_folder"]) - set(pd_all["class_folder"])
print("PlantVillage folders not found:", sorted(missing_pv))
print("PlantDoc folders not found:    ", sorted(missing_pd))

assert not missing_pv and not missing_pd, "Fix the spelling in class_alignment.csv, or check PV_CLASS_LEVEL / PD_CLASS_LEVEL in Step 1"
assert not alignment.duplicated(["canonical"]).any(), "Each class should appear once"
print(f"\nOK: {len(alignment)} aligned classes (reference study: 21)")

PlantVillage folders not found: []
PlantDoc folders not found:     []

OK: 21 aligned classes (reference study: 21)


## Step 4: Split PlantVillage into train / validation / test

PlantVillage is the *source* domain (clean lab images). We keep only the aligned classes, then split:

* **70% train**: the model learns from these
* **15% validation**: used to stop training at the right time, and for Setting A calibration
* **15% test**: the final in-domain score, never touched while developing

*Stratified* means each class is split in the same proportions, so a small class is not left out of a split.

In [9]:
# Keep only PlantVillage images from aligned classes, and give them the canonical class name
pv_name_to_class = dict(zip(alignment["pv_folder"], alignment["canonical"]))
pv = pv_all[pv_all["class_folder"].isin(pv_name_to_class)].copy()
pv["class"] = pv["class_folder"].map(pv_name_to_class)

# Step 1: carve off 30% (this becomes validation + test)
train_ids, rest_ids = train_test_split(
    pv.index, test_size=0.30, stratify=pv["class"], random_state=PV_SEED)

# Step 2: split that 30% in half (15% validation, 15% test)
val_ids, test_ids = train_test_split(
    rest_ids, test_size=0.50, stratify=pv.loc[rest_ids, "class"], random_state=PV_SEED)

pv["role"] = ""
pv.loc[train_ids, "role"] = "pv_train"
pv.loc[val_ids, "role"] = "pv_val"
pv.loc[test_ids, "role"] = "pv_test"

print(pv["role"].value_counts(normalize=True).round(3))

role
pv_train    0.70
pv_test     0.15
pv_val      0.15
Name: proportion, dtype: float64


In [10]:
# Per-class check: every class should appear in every split
pv.groupby(["class", "role"]).size().unstack()

role,pv_test,pv_train,pv_val
class,,,
Apple|Apple Scab,94,441,95
Apple|Cedar Apple Rust,41,193,41
Apple|Healthy,247,1151,247
Corn|Cercospora Leaf Spot Gray Leaf Spot,77,359,77
Corn|Common Rust,179,834,179
Corn|Northern Leaf Blight,147,690,148
Grape|Black Rot,177,826,177
Grape|Healthy,63,296,64
Peach|Healthy,54,252,54


## Step 5: Separate PlantDoc into *known* and *unknown*

PlantDoc is the *target* domain (real field photos).

* **Known:** classes that also exist in PlantVillage. The model can legitimately predict these.
* **Unknown:** PlantDoc classes with no PlantVillage match. The model has never seen them. We keep them
  aside and use them only at the very end, to test whether the model is wrongly confident on diseases it
  doesn't know.

In [11]:
pd_name_to_class = dict(zip(alignment["pd_folder"], alignment["canonical"]))

pdoc = pd_all.copy()
is_known = pdoc["class_folder"].isin(pd_name_to_class)

pdoc["role"] = np.where(is_known, "pd_known", "pd_unknown")
pdoc["class"] = np.where(is_known, pdoc["class_folder"].map(pd_name_to_class), pdoc["class_folder"])

print(pdoc["role"].value_counts())
print("Unknown categories:", pdoc.loc[~is_known, "class"].nunique(), "(reference study: 7)")
print("Reference counts: 2081 known, 490 unknown")

role
pd_known      2018
pd_unknown     466
Name: count, dtype: int64
Unknown categories: 7 (reference study: 7)
Reference counts: 2081 known, 490 unknown


### Why we pool PlantDoc's `train` and `test` folders

PlantDoc's creators split their images into `train/` and `test/` for their own experiments. **We ignore that
split.** In this project the roles are defined by our own settings: Setting A uses *all* known PlantDoc
images as the field test set, and Setting B carves out a calibration set and a test set itself.
So we pool both folders into one PlantDoc pool and keep the original folder name only in the
`original_subset` column, for traceability.

One consequence: the same class has two folders (`train/<class>` and `test/<class>`). Because we read the class
name from the class folder, both end up under the same class, which is what we want.
(The reference paper does not say how it treated this split, so we are making a documented choice
of our own, not copying one.)

In [12]:
# Same class, two subsets: confirm that train and test images were pooled under one class name
pdoc.groupby(["class", "original_subset"]).size().unstack(fill_value=0).head(10)

original_subset,test,train
class,,
Apple|Apple Scab,10,78
Apple|Cedar Apple Rust,10,76
Apple|Healthy,9,78
Bell_pepper leaf,7,49
Bell_pepper leaf spot,9,54
Blueberry leaf,11,98
Cherry leaf,10,46
Corn|Cercospora Leaf Spot Gray Leaf Spot,3,61
Corn|Common Rust,10,102


### A note on "groups" (why we need this column)

In Setting B we cut a calibration set out of PlantDoc. If two crops come from the **same original photo**
and one lands in calibration while the other lands in the test set, the test set is slightly "contaminated".
To avoid that, splitting is done by **group** (one group = one original photo), so a photo never gets split.

The reference study checked this and found each crop came from its own photo. For now we follow the same
assumption: **every crop is its own group**. This is simple and matches what the study effectively had.
(We can revisit this later if we want a stronger check.)

In [13]:
pdoc["group"] = pdoc["path"]    # each crop is its own group
pdoc["group"].is_unique

True

## Step 6: Build Setting A

Setting A pretends we have **no labelled field images**.

* **Calibration set:** the PlantVillage validation images
* **Field test set:** *all* known PlantDoc images

Nothing needs to be split here, we just gather the right rows.

In [14]:
known = pdoc[pdoc["role"] == "pd_known"].copy()

setting_A = {
    "calibration": pv[pv["role"] == "pv_val"],
    "field_test":  known,
}

for name, table in setting_A.items():
    print(f"Setting A {name:12s}: {len(table)} images")

Setting A calibration : 3986 images
Setting A field_test  : 2018 images


## Step 7: Build Setting B

Setting B pretends we can afford to label a **small sample of field images** (about 10%).

* **Calibration set:** ~10% of known PlantDoc images
* **Field test set:** the other ~90%

We use `StratifiedGroupKFold` with 10 folds. It cuts the data into 10 roughly equal pieces, keeping:
* **stratified**: class proportions similar in every piece
* **group**: a group never straddles two pieces

We simply take **one piece (~10%) as calibration** and the other nine as test. Changing the random seed
changes *which* piece, so repeating over seeds 42 to 51 shows how much our results depend on that luck.

In [15]:
# First, just ONE seed so we can inspect it carefully
seed = 42
n_folds = round(1 / CAL_FRAC)    # 10 folds -> each piece is ~10%

splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)

# .split() yields (train_positions, test_positions) for each fold; we only want the first fold's "test" piece
_, cal_positions = next(splitter.split(known, known["class"], known["group"]))

cal = known.iloc[cal_positions]
test = known.drop(cal.index)

print(f"Seed {seed}: calibration = {len(cal)}, test = {len(test)}  (reference: about 199 / 1882)")

Seed 42: calibration = 202, test = 1816  (reference: about 199 / 1882)


*If you see a scikit-learn warning that "the least populated class has only N members, less than n_splits",
that is expected when a PlantDoc class is small. It means that class has fewer images than folds, so it may
not appear in every calibration set. Step 8 checks this.*

In [16]:
# How are the classes spread across calibration and test for this seed?
counts = pd.DataFrame({
    "calibration": cal["class"].value_counts(),
    "test": test["class"].value_counts(),
}).fillna(0).astype(int)
counts

,calibration,test
class,,
Apple|Apple Scab,9,79
Apple|Cedar Apple Rust,9,77
Apple|Healthy,8,79
Corn|Cercospora Leaf Spot Gray Leaf Spot,6,58
Corn|Common Rust,11,101
Corn|Northern Leaf Blight,19,167
Grape|Black Rot,6,54
Grape|Healthy,6,58
Peach|Healthy,11,99


Now repeat for all 10 seeds. For each seed we add one column to the table, `B_seed42`, `B_seed43`, ...,
holding either `"cal"` or `"test"` for every known PlantDoc image.

In [17]:
for seed in SEEDS:
    splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    _, cal_positions = next(splitter.split(known, known["class"], known["group"]))

    column = f"B_seed{seed}"
    known[column] = "test"                                          # default: everything is test
    known.iloc[cal_positions, known.columns.get_loc(column)] = "cal"  # the chosen piece is calibration

B_columns = [c for c in known.columns if c.startswith("B_seed")]
summary = pd.DataFrame({c: known[c].value_counts() for c in B_columns}).T
summary

,test,cal
B_seed42,1816,202
B_seed43,1816,202
B_seed44,1816,202
B_seed45,1816,202
B_seed46,1815,203
B_seed47,1816,202
B_seed48,1815,203
B_seed49,1816,202
B_seed50,1816,202
B_seed51,1816,202


## Step 8: Safety checks

These checks stop us from silently building experiments on a leaky split. If one fails, an error message
tells you what to fix. Passing means the splits behave as intended.

In [18]:
all_classes = set(pv["class"])

# 1. PlantVillage: every class appears in train, val and test
for role in ["pv_train", "pv_val", "pv_test"]:
    assert set(pv.loc[pv["role"] == role, "class"]) == all_classes, f"A class is missing from {role}"

# 2. PlantDoc known classes are exactly the PlantVillage classes
assert set(known["class"]) == all_classes

# 3. For every seed: no group appears in both calibration and test, and every class has test images
for col in B_columns:
    cal_groups = set(known.loc[known[col] == "cal", "group"])
    test_groups = set(known.loc[known[col] == "test", "group"])
    assert not (cal_groups & test_groups), f"{col}: a photo is in both calibration and test"
    assert set(known.loc[known[col] == "test", "class"]) == all_classes, f"{col}: a class has no test images"

# 4. Different seeds should give different calibration sets
cal_sets = {frozenset(known.index[known[c] == "cal"]) for c in B_columns}
assert len(cal_sets) > 1, "All seeds produced the same calibration set"

print("All safety checks passed.")

All safety checks passed.


In [19]:
# Information only: which classes are missing from the calibration set in some seed?
for col in B_columns:
    absent = all_classes - set(known.loc[known[col] == "cal", "class"])
    if absent:
        print(f"{col}: no calibration images for {sorted(absent)}")
print("(Nothing printed above means every class appears in every calibration set.)")

(Nothing printed above means every class appears in every calibration set.)


## Step 9: Save the table

We put everything into **one table** with one row per image, and save it. Later notebooks only read this file.

| column | meaning |
|---|---|
| `path`, `class_folder`, `original_subset` | where the image is, its class folder name, and the `train`/`test` folder it came from (PlantDoc only) |
| `class` | the canonical class name |
| `role` | `pv_train`, `pv_val`, `pv_test`, `pd_known`, `pd_unknown` |
| `group` | original-photo group |
| `B_seed42` ... | `cal` or `test` (Setting B, known PlantDoc only) |

In [20]:
# Bring the Setting B columns back onto the full PlantDoc table (unknown images get blanks)
pdoc_full = pdoc.join(known[B_columns])
pdoc_full[B_columns] = pdoc_full[B_columns].fillna("")

pv_out = pv.assign(group=pv["path"])           # PlantVillage is split image by image
pv_out["dataset"] = "pv"
pv_out["original_subset"] = ""
pdoc_full["dataset"] = "pd"

columns = ["dataset", "path", "class_folder", "original_subset", "class", "role", "group"] + B_columns
manifest = pd.concat([pv_out, pdoc_full], ignore_index=True)[columns]
manifest[B_columns] = manifest[B_columns].fillna("")

# Safety check: unknown images must never be assigned to any calibration/test split
assert (manifest.loc[manifest["role"] == "pd_unknown", B_columns] == "").all().all()

manifest.to_csv(OUT_DIR / "master_manifest.csv", index=False)
print("Saved", OUT_DIR / "master_manifest.csv", "with", len(manifest), "rows")
manifest["role"].value_counts()

Saved outputs\master_manifest.csv with 29058 rows


role
pv_train      18601
pv_test        3987
pv_val         3986
pd_known       2018
pd_unknown      466
Name: count, dtype: int64

## What we did and what comes next

* We matched the two datasets' class names and kept the shared classes.
* We split PlantVillage into train / validation / test.
* We set aside the unknown PlantDoc classes.
* **Setting A:** calibration = PlantVillage validation, test = all known PlantDoc.
* **Setting B:** a ~10% PlantDoc calibration set and ~90% test set, repeated over 10 seeds.
* We checked there is no leakage and saved everything to `master_manifest.csv`.

**Next notebook:** load this manifest, fine-tune the ResNet-50 on `pv_train`, and measure accuracy on
PlantVillage and PlantDoc. That's where we first see the reliability gap for ourselves.